# LibrarySphere — Smart Library Management System
**Full-stack web development capstone | Google Colab**

A database-driven library portal using Python, Flask, SQLite, HTML and CSS. It follows the architecture of a traditional Flask management system, but manages books and borrowing rather than student records.

### Features
- Book inventory: add, edit, delete and view books.
- Issue and return books, with automatic stock updates.
- Borrower details and due dates.
- Dashboard: titles, total copies, issued copies and overdue loans.
- Search and sorting by title, author, category and stock.
- Low-stock and overdue indicators.
- Popular-book ranking.
- CSV export for inventory and loans.
- SQLite database and sample records.
- Responsive design and data analysis using Pandas/Matplotlib.

**Architecture:** HTML/CSS → Flask routes → SQLite.

Educational prototype only. Colab is temporary and this version has no staff authentication or production-grade security. Use fictional borrower data.

## 1. Install packages

In [ ]:
!pip -q install flask pandas matplotlib seaborn

## 2. Imports and folders

In [ ]:
import os, sqlite3, csv, io, time, subprocess
from datetime import date, datetime
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from flask import Flask, render_template, request, redirect, url_for, Response, flash

BASE = Path('/content/librarysphere')
(BASE/'templates').mkdir(parents=True, exist_ok=True)
(BASE/'static').mkdir(parents=True, exist_ok=True)
os.chdir(BASE)
DB_NAME = 'library.db'
print('Workspace:', BASE)

## 3. Create the SQLite database

In [ ]:
def get_db():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    conn.execute('PRAGMA foreign_keys=ON')
    return conn

def init_db():
    with get_db() as conn:
        conn.execute('''
        CREATE TABLE IF NOT EXISTS books(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL, author TEXT NOT NULL,
            isbn TEXT NOT NULL UNIQUE, category TEXT NOT NULL,
            total_copies INTEGER NOT NULL CHECK(total_copies>=0),
            available_copies INTEGER NOT NULL CHECK(available_copies>=0),
            created_at TEXT NOT NULL
        )''')
        conn.execute('''
        CREATE TABLE IF NOT EXISTS loans(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            book_id INTEGER NOT NULL,
            borrower_name TEXT NOT NULL, borrower_email TEXT NOT NULL,
            issue_date TEXT NOT NULL, due_date TEXT NOT NULL,
            return_date TEXT,
            status TEXT NOT NULL CHECK(status IN ('Issued','Returned')),
            FOREIGN KEY(book_id) REFERENCES books(id) ON DELETE RESTRICT
        )''')
init_db()
print('Database created.')

## 4. Add demonstration books

In [ ]:
demo_books = [
 ('Atomic Habits','James Clear','9780735211292','Self Development',5),
 ('The Alchemist','Paulo Coelho','9780061122415','Fiction',4),
 ('Clean Code','Robert C. Martin','9780132350884','Technology',3),
 ('Deep Work','Cal Newport','9781455586691','Productivity',4),
 ('The Psychology of Money','Morgan Housel','9780857197689','Finance',2),
 ('Python Crash Course','Eric Matthes','9781718502703','Technology',3),
 ('Pride and Prejudice','Jane Austen','9780141439518','Classic',2),
 ('Ikigai','Hector Garcia','9780143130727','Self Development',3)
]
with get_db() as conn:
    if conn.execute('SELECT COUNT(*) FROM books').fetchone()[0] == 0:
        conn.executemany('''
        INSERT INTO books(title,author,isbn,category,total_copies,available_copies,created_at)
        VALUES(?,?,?,?,?,?,?)''',
        [(t,a,i,c,n,n,datetime.now().isoformat(timespec='seconds')) for t,a,i,c,n in demo_books])
print('Sample records loaded if the database was empty.')

## 5. Create the Flask backend

In [ ]:
app = Flask(__name__)
app.secret_key = 'librarysphere-colab-demo'

@app.route('/', methods=['GET','POST'])
def index():
    if request.method == 'POST':
        fields = [request.form.get(k,'').strip() for k in ('title','author','isbn','category')]
        try:
            copies = int(request.form.get('total_copies',''))
        except ValueError:
            flash('Copies must be a whole number.','error')
            return redirect(url_for('index'))
        if not all(fields) or copies < 1:
            flash('Complete all fields and enter at least one copy.','error')
            return redirect(url_for('index'))
        try:
            with get_db() as conn:
                conn.execute('''INSERT INTO books(title,author,isbn,category,total_copies,available_copies,created_at)
                    VALUES(?,?,?,?,?,?,?)''', (*fields,copies,copies,datetime.now().isoformat(timespec='seconds')))
            flash('Book added successfully.','success')
        except sqlite3.IntegrityError:
            flash('This ISBN already exists.','error')
        return redirect(url_for('index'))

    search = request.args.get('search','').strip()
    sort = request.args.get('sort','title')
    order = 'DESC' if request.args.get('order','asc').lower() == 'desc' else 'ASC'
    allowed = {'title':'title COLLATE NOCASE','author':'author COLLATE NOCASE',
               'category':'category COLLATE NOCASE','copies':'total_copies','available':'available_copies'}
    sort_sql = allowed.get(sort,'title COLLATE NOCASE')
    with get_db() as conn:
        books = conn.execute(f'''SELECT * FROM books
          WHERE title LIKE ? OR author LIKE ? OR isbn LIKE ? OR category LIKE ?
          ORDER BY {sort_sql} {order}, id''', (f'%{search}%',)*4).fetchall()
        metrics = conn.execute('''SELECT COUNT(*) titles, COALESCE(SUM(total_copies),0) copies,
          COALESCE(SUM(total_copies-available_copies),0) issued,
          COALESCE(SUM(available_copies),0) available FROM books''').fetchone()
        overdue = conn.execute("SELECT COUNT(*) FROM loans WHERE status='Issued' AND due_date < ?",
                               (date.today().isoformat(),)).fetchone()[0]
        popular = conn.execute('''SELECT b.title,b.author,COUNT(l.id) issue_count
          FROM books b LEFT JOIN loans l ON b.id=l.book_id
          GROUP BY b.id ORDER BY issue_count DESC,b.title LIMIT 5''').fetchall()
        loans = conn.execute('''SELECT l.*,b.title FROM loans l JOIN books b ON b.id=l.book_id
          ORDER BY CASE WHEN l.status='Issued' THEN 0 ELSE 1 END,l.due_date LIMIT 15''').fetchall()
    return render_template('index.html',books=books,metrics=dict(metrics),overdue=overdue,
       popular=popular,loans=loans,search=search,sort=sort,order=order,today=date.today().isoformat())

@app.route('/edit/<int:book_id>', methods=['GET','POST'])
def edit_book(book_id):
    with get_db() as conn:
        book=conn.execute('SELECT * FROM books WHERE id=?',(book_id,)).fetchone()
    if book is None:
        flash('Book not found.','error')
        return redirect(url_for('index'))
    if request.method == 'POST':
        title,author,isbn,category=[request.form.get(k,'').strip() for k in ('title','author','isbn','category')]
        try:
            total=int(request.form.get('total_copies',''))
        except ValueError:
            flash('Enter a valid copy count.','error')
            return redirect(url_for('edit_book',book_id=book_id))
        issued=book['total_copies']-book['available_copies']
        if not all([title,author,isbn,category]) or total < max(1,issued):
            flash(f'Complete all fields. Total copies cannot be below {issued} issued copies.','error')
            return redirect(url_for('edit_book',book_id=book_id))
        try:
            with get_db() as conn:
                conn.execute('''UPDATE books SET title=?,author=?,isbn=?,category=?,total_copies=?,available_copies=? WHERE id=?''',
                  (title,author,isbn,category,total,total-issued,book_id))
            flash('Book updated.','success')
            return redirect(url_for('index'))
        except sqlite3.IntegrityError:
            flash('That ISBN is already in use.','error')
    return render_template('edit.html',book=book)

@app.post('/delete/<int:book_id>')
def delete_book(book_id):
    with get_db() as conn:
        active=conn.execute("SELECT COUNT(*) FROM loans WHERE book_id=? AND status='Issued'",(book_id,)).fetchone()[0]
        if active:
            flash('Return all issued copies before deleting this book.','error')
        else:
            conn.execute('DELETE FROM books WHERE id=?',(book_id,))
            flash('Book deleted.','success')
    return redirect(url_for('index'))

@app.post('/issue')
def issue_book():
    try:
        book_id=int(request.form.get('book_id',''))
        borrower=request.form.get('borrower_name','').strip()
        email=request.form.get('borrower_email','').strip()
        due=request.form.get('due_date','')
        datetime.strptime(due,'%Y-%m-%d')
    except (ValueError,TypeError):
        flash('Enter valid issue details and due date.','error')
        return redirect(url_for('index'))
    if not borrower or not email or due < date.today().isoformat():
        flash('Borrower details are required; due date cannot be in the past.','error')
        return redirect(url_for('index'))
    with get_db() as conn:
        book=conn.execute('SELECT * FROM books WHERE id=?',(book_id,)).fetchone()
        if book is None or book['available_copies'] < 1:
            flash('No available copy for the selected book.','error')
        else:
            conn.execute('UPDATE books SET available_copies=available_copies-1 WHERE id=?',(book_id,))
            conn.execute('''INSERT INTO loans(book_id,borrower_name,borrower_email,issue_date,due_date,status)
                            VALUES(?,?,?,?,?,'Issued')''',
                         (book_id,borrower,email,date.today().isoformat(),due))
            flash('Book issued successfully.','success')
    return redirect(url_for('index'))

@app.post('/return/<int:loan_id>')
def return_book(loan_id):
    with get_db() as conn:
        loan=conn.execute("SELECT * FROM loans WHERE id=? AND status='Issued'",(loan_id,)).fetchone()
        if loan is None:
            flash('Active loan not found.','error')
        else:
            conn.execute("UPDATE loans SET status='Returned',return_date=? WHERE id=?",
                         (date.today().isoformat(),loan_id))
            conn.execute('UPDATE books SET available_copies=available_copies+1 WHERE id=?',(loan['book_id'],))
            flash('Book returned successfully.','success')
    return redirect(url_for('index'))

@app.get('/export/books')
def export_books():
    with get_db() as conn:
        rows=conn.execute('SELECT * FROM books ORDER BY title').fetchall()
    out=io.StringIO(); writer=csv.writer(out)
    writer.writerow(['ID','Title','Author','ISBN','Category','Total Copies','Available Copies'])
    for r in rows:
        writer.writerow([r['id'],r['title'],r['author'],r['isbn'],r['category'],r['total_copies'],r['available_copies']])
    return Response('\ufeff'+out.getvalue(),mimetype='text/csv; charset=utf-8',
       headers={'Content-Disposition':'attachment; filename=library_inventory.csv'})

@app.get('/export/loans')
def export_loans():
    with get_db() as conn:
        rows=conn.execute('''SELECT l.id,b.title,l.borrower_name,l.borrower_email,l.issue_date,l.due_date,l.return_date,l.status
          FROM loans l JOIN books b ON b.id=l.book_id ORDER BY l.id DESC''').fetchall()
    out=io.StringIO(); writer=csv.writer(out)
    writer.writerow(['Loan ID','Book','Borrower','Email','Issue Date','Due Date','Return Date','Status'])
    for r in rows: writer.writerow(list(r))
    return Response('\ufeff'+out.getvalue(),mimetype='text/csv; charset=utf-8',
       headers={'Content-Disposition':'attachment; filename=library_loans.csv'})

init_db()
print('Flask routes are ready.')

## 6. Create the main HTML dashboard

In [ ]:
index_html = r"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>LibrarySphere | Library Management</title>
<link rel="stylesheet" href="{{ url_for('static',filename='style.css') }}"></head>
<body><div class="shell">
<header class="hero"><div>
<div class="brand"><span class="logo">L</span> LIBRARY<span class="brand-light">SPHERE</span></div>
<p class="kicker">SMART LIBRARY MANAGEMENT</p><h1>Your library,<br><em>beautifully organised.</em></h1>
<p class="hero-copy">Manage inventory, track borrowing and keep your collection moving.</p></div>
<div class="hero-art"><div class="books"><i></i><i></i><i></i><i></i></div><span>READ · LEARN · GROW</span></div></header>
{% with messages=get_flashed_messages(with_categories=true) %}{% for category,message in messages %}<div class="message {{ category }}">{{ message }}</div>{% endfor %}{% endwith %}
<section class="metrics">
<article><span>BOOK TITLES</span><strong>{{ metrics.titles }}</strong><small>Unique catalogue entries</small></article>
<article><span>TOTAL COPIES</span><strong>{{ metrics.copies }}</strong><small>Across the collection</small></article>
<article><span>ON LOAN</span><strong>{{ metrics.issued }}</strong><small>Currently borrowed</small></article>
<article class="alert"><span>OVERDUE LOANS</span><strong>{{ overdue }}</strong><small>Require follow-up</small></article></section>
<section class="panel"><div class="heading"><div><p class="eyebrow">COLLECTION MANAGEMENT</p><h2>Add a new book</h2></div><span class="step">01 / INVENTORY</span></div>
<form method="POST" class="book-form">
<label>Book title<input name="title" placeholder="e.g. The Design of Everyday Things" required></label>
<label>Author<input name="author" placeholder="Author name" required></label>
<label>ISBN<input name="isbn" placeholder="Unique ISBN" required></label>
<label>Category<input name="category" placeholder="e.g. Technology" required></label>
<label>Copies<input name="total_copies" type="number" min="1" value="1" required></label>
<button class="btn primary">＋ Add to library</button></form></section>
<section class="panel"><div class="heading"><div><p class="eyebrow">YOUR COLLECTION</p><h2>Book catalogue</h2></div><a class="btn export" href="{{ url_for('export_books') }}">↓ Export inventory</a></div>
<form method="GET" class="toolbar"><input name="search" value="{{ search }}" placeholder="Search title, author, ISBN or category">
<select name="sort"><option value="title" {% if sort=='title' %}selected{% endif %}>Title</option><option value="author" {% if sort=='author' %}selected{% endif %}>Author</option><option value="category" {% if sort=='category' %}selected{% endif %}>Category</option><option value="copies" {% if sort=='copies' %}selected{% endif %}>Total copies</option><option value="available" {% if sort=='available' %}selected{% endif %}>Available copies</option></select>
<select name="order"><option value="asc" {% if order=='asc' %}selected{% endif %}>Ascending</option><option value="desc" {% if order=='desc' %}selected{% endif %}>Descending</option></select>
<button class="btn primary">Search / Sort</button><a class="btn muted-btn" href="{{ url_for('index') }}">Reset</a></form>
<div class="table-wrap"><table><thead><tr><th>Book</th><th>ISBN</th><th>Category</th><th>Copies</th><th>Availability</th><th>Actions</th></tr></thead><tbody>
{% for b in books %}<tr><td><b>{{ b.title }}</b><small>{{ b.author }}</small></td><td>{{ b.isbn }}</td><td><span class="category">{{ b.category }}</span></td>
<td>{{ b.available_copies }}<small>of {{ b.total_copies }} available</small></td>
<td><span class="pill {% if b.available_copies==0 %}unavailable{% elif b.available_copies<=1 %}low{% endif %}">{% if b.available_copies==0 %}Unavailable{% elif b.available_copies<=1 %}Low stock{% else %}Available{% endif %}</span></td>
<td class="actions"><a class="mini" href="{{ url_for('edit_book',book_id=b.id) }}">Edit</a><form method="POST" action="{{ url_for('delete_book',book_id=b.id) }}" onsubmit="return confirm('Delete this book?')"><button class="mini danger">Delete</button></form></td></tr>
{% else %}<tr><td colspan="6" class="empty">No books found. Add your first book above.</td></tr>{% endfor %}</tbody></table></div><p class="footnote">Showing {{ books|length }} matching records.</p></section>
<section class="two-col"><section class="panel"><div class="heading"><div><p class="eyebrow">CIRCULATION</p><h2>Issue a book</h2></div><span class="step">02 / BORROW</span></div>
<form method="POST" action="{{ url_for('issue_book') }}" class="stack">
<label>Select book<select name="book_id" required>{% for b in books if b.available_copies>0 %}<option value="{{ b.id }}">{{ b.title }} — {{ b.available_copies }} available</option>{% endfor %}</select></label>
<label>Borrower name<input name="borrower_name" required placeholder="Full name"></label><label>Borrower email<input name="borrower_email" type="email" required placeholder="name@example.com"></label>
<label>Due date<input name="due_date" type="date" min="{{ today }}" required></label><button class="btn primary">Issue book →</button></form></section>
<section class="panel"><div class="heading"><div><p class="eyebrow">COLLECTION INSIGHT</p><h2>Popular titles</h2></div></div>
{% for p in popular %}<div class="rank"><span>0{{ loop.index }}</span><div><b>{{ p.title }}</b><small>{{ p.author }}</small></div><em>{{ p.issue_count }} loans</em></div>{% else %}<p class="empty">No borrowing activity yet.</p>{% endfor %}</section></section>
<section class="panel"><div class="heading"><div><p class="eyebrow">ACTIVE CIRCULATION</p><h2>Recent loan activity</h2></div><a class="btn export" href="{{ url_for('export_loans') }}">↓ Export loans</a></div>
<div class="table-wrap"><table><thead><tr><th>Book</th><th>Borrower</th><th>Issue date</th><th>Due date</th><th>Status</th><th>Action</th></tr></thead><tbody>
{% for l in loans %}<tr><td><b>{{ l.title }}</b></td><td>{{ l.borrower_name }}<small>{{ l.borrower_email }}</small></td><td>{{ l.issue_date }}</td><td>{{ l.due_date }}</td>
<td><span class="pill {% if l.status=='Issued' and l.due_date<today %}unavailable{% elif l.status=='Returned' %}returned{% endif %}">{% if l.status=='Issued' and l.due_date<today %}Overdue{% else %}{{ l.status }}{% endif %}</span></td>
<td>{% if l.status=='Issued' %}<form method="POST" action="{{ url_for('return_book',loan_id=l.id) }}"><button class="mini return">Mark returned</button></form>{% else %}<small>Returned {{ l.return_date }}</small>{% endif %}</td></tr>
{% else %}<tr><td colspan="6" class="empty">No loans recorded yet.</td></tr>{% endfor %}</tbody></table></div></section>
<footer><span>LIBRARYSPHERE · CAPSTONE PROJECT</span><span>FLASK + SQLITE</span></footer></div></body></html>"""
(Path('templates')/'index.html').write_text(index_html,encoding='utf-8')
print('Main dashboard created.')

## 7. Create the edit page

In [ ]:
edit_html = r"""<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Edit Book | LibrarySphere</title><link rel="stylesheet" href="{{ url_for('static',filename='style.css') }}"></head>
<body><div class="edit-shell"><a class="back" href="{{ url_for('index') }}">← Back to catalogue</a><section class="panel">
<p class="eyebrow">INVENTORY MANAGEMENT</p><h1>Edit book details</h1>
{% with messages=get_flashed_messages(with_categories=true) %}{% for category,message in messages %}<div class="message {{ category }}">{{ message }}</div>{% endfor %}{% endwith %}
<form method="POST" class="stack">
<label>Book title<input name="title" value="{{ book.title }}" required></label><label>Author<input name="author" value="{{ book.author }}" required></label>
<label>ISBN<input name="isbn" value="{{ book.isbn }}" required></label><label>Category<input name="category" value="{{ book.category }}" required></label>
<label>Total copies<input name="total_copies" type="number" min="1" value="{{ book.total_copies }}" required></label>
<p class="footnote">Currently issued: {{ book.total_copies-book.available_copies }}. Total copies cannot be reduced below this number.</p>
<button class="btn primary">Save changes</button><a class="btn muted-btn" href="{{ url_for('index') }}">Cancel</a></form></section></div></body></html>"""
(Path('templates')/'edit.html').write_text(edit_html,encoding='utf-8')
print('Edit page created.')

## 8. Create the CSS stylesheet

In [ ]:
css = r"""@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Playfair+Display:ital,wght@0,600;0,700;1,600&display=swap');
:root{--bg:#f4f2ed;--paper:#fffefa;--ink:#222a25;--muted:#7c847d;--line:#e8e5dc;--green:#315b49;--sage:#e9eee8;--gold:#b58a52;--red:#a64d45}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font-family:'DM Sans',sans-serif;padding:28px 4%}.shell{max-width:1320px;margin:auto}
.hero{background:#1f3d32;color:white;border-radius:20px;padding:35px 42px;display:flex;justify-content:space-between;align-items:center;min-height:255px;overflow:hidden}
.brand{font-size:14px;letter-spacing:2px;font-weight:700;display:flex;align-items:center;gap:9px}.logo{width:34px;height:34px;border:1px solid #9fb7a6;border-radius:50%;display:grid;place-items:center;font:700 19px 'Playfair Display'}.brand-light{font-weight:400;color:#b8c9bd}
.kicker,.eyebrow{font-size:9px;letter-spacing:2px;font-weight:700;color:var(--gold);margin:23px 0 8px}.hero h1{font:700 39px 'Playfair Display';line-height:1.12;margin:0}.hero h1 em{color:#c9d8c9}.hero-copy{font-size:12px;color:#d0d9d1;margin-top:13px}
.hero-art{width:190px;height:170px;border:1px solid #ffffff30;border-radius:50%;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:15px;margin-right:6%}.hero-art span{font-size:7px;letter-spacing:2px;color:#cbd7ce}.books{height:85px;display:flex;align-items:end;gap:5px}.books i{width:26px;height:65px;background:#b78d5c;border:1px solid #ffffff70;border-radius:4px 4px 1px 1px}.books i:nth-child(2){height:82px;background:#d5c6a5}.books i:nth-child(3){height:72px;background:#6c8a73}.books i:nth-child(4){height:59px;background:#a85e4f}
.metrics{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:18px 0}.metrics article,.panel{background:var(--paper);border:1px solid var(--line);border-radius:14px;box-shadow:0 10px 30px #26382c0a}.metrics article{padding:18px 20px}.metrics span{font-size:9px;letter-spacing:1.4px;color:var(--muted);font-weight:700}.metrics strong{display:block;font:700 29px 'Playfair Display';margin:9px 0 3px}.metrics small,td small{font-size:9px;color:var(--muted)}.metrics .alert strong{color:var(--red)}
.panel{padding:22px;margin-bottom:17px}.heading{display:flex;align-items:center;justify-content:space-between;gap:12px;margin-bottom:18px}.heading .eyebrow{margin:0 0 6px}.heading h2,.panel h1{font:700 21px 'Playfair Display';margin:0}.step{font-size:9px;letter-spacing:1.2px;color:#9ba39a}
.book-form{display:grid;grid-template-columns:1.5fr 1.2fr 1fr 1fr .7fr auto;gap:10px;align-items:end}.book-form label,.stack label{display:grid;gap:6px;font-size:10px;color:var(--muted);font-weight:700}
input,select{width:100%;min-width:0;border:1px solid var(--line);border-radius:7px;background:white;color:var(--ink);padding:11px 10px;font:12px 'DM Sans'}input:focus,select:focus{outline:2px solid #b8c9bd}
.btn{border:0;border-radius:7px;padding:11px 13px;font:600 11px 'DM Sans';text-decoration:none;display:inline-flex;justify-content:center;align-items:center;cursor:pointer;white-space:nowrap}.primary{background:var(--green);color:white}.export{background:var(--sage);color:var(--green)}.muted-btn{background:#f0eee8;color:#626b63}
.toolbar{display:flex;gap:9px;margin-bottom:15px;flex-wrap:wrap}.toolbar input{flex:1;min-width:190px}.toolbar select{width:auto;min-width:125px}
.table-wrap{overflow-x:auto}table{width:100%;border-collapse:collapse;min-width:760px}th{text-align:left;padding:11px 10px;color:#858d84;font-size:9px;letter-spacing:1.2px;text-transform:uppercase;border-bottom:1px solid var(--line)}td{padding:13px 10px;font-size:11px;border-bottom:1px solid #eeece5}td small{display:block;margin-top:4px}
.category{background:#f0eee6;border-radius:20px;padding:5px 8px;color:#696f64;font-size:9px}.pill{display:inline-block;border-radius:20px;padding:5px 8px;font-size:9px;font-weight:700;background:#e6f0e8;color:#38644d}.pill.low{background:#fff1dc;color:#a36d2c}.pill.unavailable{background:#f9e8e5;color:#a64d45}.pill.returned{background:#e9eee8;color:#526c58}
.actions{display:flex;gap:5px;align-items:center}.actions form{margin:0}.mini{border:0;border-radius:5px;padding:7px 9px;background:#e9eee8;color:#315b49;font:600 9px 'DM Sans';text-decoration:none;cursor:pointer;white-space:nowrap}.mini.danger{background:#f8e9e6;color:#a64d45}.mini.return{background:#315b49;color:white}
.footnote{font-size:10px;color:var(--muted);margin-top:13px}.two-col{display:grid;grid-template-columns:1fr 1fr;gap:17px}.stack{display:grid;gap:12px}.stack .btn{justify-self:start}.rank{display:flex;align-items:center;gap:12px;padding:13px 0;border-bottom:1px solid var(--line)}.rank>span{font:italic 17px 'Playfair Display';color:var(--gold)}.rank b{font-size:11px}.rank small{display:block;font-size:9px;color:var(--muted);margin-top:4px}.rank em{margin-left:auto;font-size:9px;color:var(--muted)}.empty{text-align:center;color:var(--muted);padding:20px}
.message{padding:11px 14px;border-radius:8px;font-size:11px;margin:12px 0}.success{background:#e6f0e8;color:#315b49}.error{background:#f9e8e5;color:#a64d45}footer{display:flex;justify-content:space-between;padding:9px 2px;color:#8a9188;font-size:9px;letter-spacing:1px}
.edit-shell{max-width:650px;margin:25px auto}.back{display:inline-block;color:var(--green);text-decoration:none;font-size:12px;margin:8px 0 18px}
@media(max-width:1050px){.book-form{grid-template-columns:repeat(3,1fr)}.metrics{grid-template-columns:repeat(2,1fr)}}
@media(max-width:700px){body{padding:12px}.hero{padding:25px;min-height:225px}.hero h1{font-size:31px}.hero-art{width:125px;height:125px;margin:0}.books{transform:scale(.75)}.hero-art span{font-size:6px}.panel{padding:15px}.book-form{grid-template-columns:1fr 1fr}.two-col{grid-template-columns:1fr}.toolbar input{min-width:100%}.toolbar select{flex:1}.metrics{gap:8px}.metrics article{padding:14px}}
@media(max-width:430px){.hero-art{display:none}.book-form{grid-template-columns:1fr}.metrics{grid-template-columns:1fr 1fr}.heading{align-items:flex-start;flex-direction:column}}
"""
(Path('static')/'style.css').write_text(css,encoding='utf-8')
print('Stylesheet created.')

## 9. Launch the Flask application in Google Colab

This starts the Flask server in a background thread and creates a Colab browser-proxy link. Keep the notebook runtime active while using the website.

In [ ]:

try:
    server.terminate()
    server.wait(timeout=3)
except Exception:
    pass

import threading
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_flask, daemon=True)
server_thread.start()
time.sleep(2)

from google.colab.output import eval_js
app_url = eval_js('google.colab.kernel.proxyPort(5000)')
print('LibrarySphere is running. Open this link:')
print(app_url)


In [ ]:
# Save the current Flask application object into a WSGI runner.
# In Colab, the current kernel can serve the app directly using Flask's development server.
try:
    server.terminate()
    server.wait(timeout=3)
except Exception:
    pass

import threading
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_flask, daemon=True)
server_thread.start()
time.sleep(2)

from google.colab.output import eval_js
app_url = eval_js('google.colab.kernel.proxyPort(5000)')
print('LibrarySphere is running. Open this link:')
print(app_url)

## 10. Inspect the database with Pandas

In [ ]:
with get_db() as conn:
    books_df = pd.read_sql_query('SELECT * FROM books ORDER BY title', conn)
    loans_df = pd.read_sql_query('SELECT * FROM loans ORDER BY id DESC', conn)
display(books_df.head(10))
print('Book titles:', len(books_df))
print('Loan records:', len(loans_df))

## 11. Visualise inventory data

In [ ]:
if not books_df.empty:
    category_totals = books_df.groupby('category')['total_copies'].sum().sort_values(ascending=False)
    plt.figure(figsize=(9,4))
    category_totals.plot(kind='bar')
    plt.title('Total Book Copies by Category')
    plt.xlabel('Category')
    plt.ylabel('Copies')
    plt.xticks(rotation=25, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
if not books_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(books_df['available_copies'], bins=8)
    plt.title('Available Copies per Book Title')
    plt.xlabel('Available copies')
    plt.tight_layout()
    plt.show()

## 12. Testing checklist
1. Add a book with a unique ISBN.
2. Try a duplicate ISBN; the application should reject it.
3. Edit a book and change its copy count.
4. Confirm total copies cannot be reduced below the number currently issued.
5. Search by title, author, ISBN and category.
6. Sort by title, author, category and stock.
7. Issue a book with borrower details and a due date.
8. Confirm available copies decrease after issue.
9. Return a book and confirm available copies increase.
10. Try deleting a book with an active loan; deletion should be blocked.
11. Export inventory and loan records as CSV.
12. Check the overdue indicator.

## 13. Conclusion and future scope
LibrarySphere demonstrates a full-stack CRUD application with relational database management and a borrowing workflow. Flask handles routes and form processing, SQLite stores books and loans, HTML structures the interface, CSS styles the dashboard, and Pandas/Matplotlib/Seaborn support inventory analysis.

**Future upgrades:** staff authentication, role-based access, barcode scanning, fine calculation, email reminders, book cover uploads, PDF receipts, a REST API and cloud database hosting.

**Limitations:** Google Colab is temporary hosting. This demonstration does not include authentication, CSRF protection or production-grade security. Do not enter real borrower information.